# 18 Transformer多任务微调与A–F消融

**目的**：在16号冻结的数据边界与17号折内自监督检查点之上，完成P/B/T五端点的
masked多任务微调，并比较六条预先定义的输入/扩增路线。

- 5个外折 × 6条路线 × 3个随机种子 = **90个独立训练run**；
- P为二分类；B和三个T端点为回归；缺失标签只在对应端点掩码，不做插补；
- 内层验证仅用于epoch选择，外层真实谱测试只在最佳epoch冻结后运行；
- 所有路线使用完全相同的真实外层测试化合物；
- ChemVista仍然锁定，不读取、不调参、不报告其结果。

## 六条预注册路线

- **A**：真实Development谱，从随机初始化训练；
- **B**：公共实测谱自监督预训练后，仅用真实Development谱微调；
- **C**：真实谱＋同化合物CFM 10/20 eV监督扩增，从随机初始化训练；
- **D**：真实谱＋训练域内“仅合成可用”化合物CFM监督扩展，从随机初始化训练；
- **E**：公共实测谱＋允许CFM自监督预训练后，仅用真实Development谱微调；
- **F**：仅CFM监督训练，作为sim-to-real负对照。

每个epoch先均匀抽取化合物，再在该化合物的候选谱中按16号冻结的`sample_weight`
选择一张谱，因此谱图多的化合物不会支配梯度。回归标准化参数只由该外折共同的
真实训练化合物计算，六路线共享，避免额外CFM化合物改变评价尺度。

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import random
import sys
import time
import warnings

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('MKL_THREADING_LAYER', 'SEQUENTIAL')
os.environ.setdefault('MPLBACKEND', 'Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    mean_absolute_error, mean_squared_error, r2_score, roc_auc_score,
)
from tqdm.auto import tqdm

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 280)
plt.style.use('seaborn-v0_8-whitegrid')

# -------------------- 用户运行配置 --------------------
RUN_TRAINING = True
RETRAIN_COMPLETED = False
RESUME_INCOMPLETE = True
REQUIRE_CUDA = False
DRY_RUN = False
# ----------------------------------------------------

ROUTES = [
    'A_real_scratch',
    'B_real_pretrain',
    'C_same_compound_cfm',
    'D_synthetic_new_cfm',
    'E_cfm_pretrain_real_finetune',
    'F_cfm_only_negative_control',
]
TASKS = {
    'P_screen_not_ready_binary': {
        'column': 'label__P_screen_not_ready_binary', 'kind': 'classification',
    },
    'B_LOGBCF_REGRESSION': {
        'column': 'label__B_LOGBCF_REGRESSION', 'kind': 'regression',
    },
    'T_EC50_MG_L_REGRESSION': {
        'column': 'label__T_EC50_MG_L_REGRESSION', 'kind': 'regression',
    },
    'T_LC50_MG_L_REGRESSION': {
        'column': 'label__T_LC50_MG_L_REGRESSION', 'kind': 'regression',
    },
    'T_NOEC_MG_L_REGRESSION': {
        'column': 'label__T_NOEC_MG_L_REGRESSION', 'kind': 'regression',
    },
}
TASK_NAMES = list(TASKS)
N_OUTER_FOLDS = 5
N_SEEDS = 3
BASE_SEED = 20260906
MAX_EPOCHS = 20
MIN_EPOCHS = 6
EARLY_STOPPING_PATIENCE = 4
BATCH_SIZE = 64
EVALUATION_BATCH_SIZE = 128
NUM_WORKERS = 0
ENCODER_LEARNING_RATE = 2e-4
HEAD_LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
USE_MIXED_PRECISION = True
MAX_VALIDATION_SPECTRA_PER_COMPOUND = 4
HEAD_DROPOUT = 0.20

MODEL_CONFIG = {
    'peak_input_dim': 4,
    'metadata_numeric_dim': 8,
    'd_model': 128,
    'n_heads': 8,
    'n_layers': 4,
    'dim_feedforward': 256,
    'dropout': 0.10,
    'projection_dim': 128,
}

if DRY_RUN:
    N_SEEDS = 1
    MAX_EPOCHS = 1
    MIN_EPOCHS = 1
    EARLY_STOPPING_PATIENCE = 1
    BATCH_SIZE = 16
    EVALUATION_BATCH_SIZE = 32
    MAX_VALIDATION_SPECTRA_PER_COMPOUND = 2

cwd = Path.cwd().resolve()
root_candidates = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((p for p in root_candidates if (p / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请手动设置ROOT。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
FREEZE_DIR = BASE / 'transformer_data_v1'
PRETRAIN_DIR = BASE / 'transformer_pretraining_v1'
OUTPUT_DIR = BASE / ('transformer_finetuning_v1_dryrun' if DRY_RUN else 'transformer_finetuning_v1')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
PREDICTION_DIR = OUTPUT_DIR / 'outer_test_predictions'
RUN_MANIFEST_DIR = OUTPUT_DIR / 'run_manifests'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for directory in [OUTPUT_DIR, CHECKPOINT_DIR, PREDICTION_DIR, RUN_MANIFEST_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PATHS = {
    'freeze_manifest': FREEZE_DIR / 'transformer_data_freeze_manifest.json',
    'freeze_qc': FREEZE_DIR / 'transformer_data_freeze_qc.csv',
    'route_spec': FREEZE_DIR / 'transformer_ablation_routes.json',
    'token_schema': FREEZE_DIR / 'transformer_token_schema.json',
    'vocab': FREEZE_DIR / 'transformer_categorical_vocab.json',
    'labels': FREEZE_DIR / 'transformer_compound_labels.csv.gz',
    'pretraining_manifest': PRETRAIN_DIR / 'transformer_pretraining_manifest.json',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_CUDA and device.type != 'cuda':
    raise RuntimeError('REQUIRE_CUDA=True，但当前PyTorch未检测到CUDA。')
amp_enabled = bool(USE_MIXED_PRECISION and device.type == 'cuda')
print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('PyTorch:', torch.__version__, '| device:', device, '| AMP:', amp_enabled)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Planned runs:', (1 if DRY_RUN else N_OUTER_FOLDS) * len(ROUTES) * N_SEEDS)

## 1. 输入冻结状态与工具函数

本节要求16号与17号均处于就绪状态，并验证A–F、标签、token缓存和预训练检查点存在。
输出采用原子替换，避免中断后留下看似完整的半文件。

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def stable_seed(*parts):
    text = '||'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(text).digest()[:4], 'little')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

def atomic_json(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temporary, path)

def atomic_csv(frame, path, compression=None):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8-sig', compression=compression)
    os.replace(temporary, path)

def save_torch_atomic(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

def as_bool(series):
    return series.astype(str).str.lower().map({'true': True, 'false': False}).fillna(False)

for name, path in PATHS.items():
    assert path.exists(), f'缺少{name}: {path}'

freeze = json.loads(PATHS['freeze_manifest'].read_text(encoding='utf-8'))
pretrain = json.loads(PATHS['pretraining_manifest'].read_text(encoding='utf-8'))
route_spec = json.loads(PATHS['route_spec'].read_text(encoding='utf-8'))
token_schema = json.loads(PATHS['token_schema'].read_text(encoding='utf-8'))
vocab = json.loads(PATHS['vocab'].read_text(encoding='utf-8'))
freeze_qc = pd.read_csv(PATHS['freeze_qc'])
assert freeze['status'] == 'READY_FOR_NOTEBOOK_17'
assert pretrain['status'] == 'READY_FOR_NOTEBOOK_18'
assert freeze['external_test_unlocked'] is False
assert freeze['chemvista_spectra_used'] is False
assert set(ROUTES) == set(route_spec['routes'])
assert bool(as_bool(freeze_qc['passed']).all())
assert token_schema['peak_token_columns'] == [
    'fragment_mz_norm', 'sqrt_relative_intensity',
    'neutral_loss_norm', 'fragment_precursor_ratio',
]

INPUT_HASHES = {name: sha256_file(path) for name, path in PATHS.items()}
labels = pd.read_csv(PATHS['labels'])
required_label_columns = ['inchikey_2d', 'cv_fold', *[x['column'] for x in TASKS.values()]]
missing = sorted(set(required_label_columns) - set(labels.columns))
assert not missing, f'标签表缺列: {missing}'
labels['inchikey_2d'] = labels['inchikey_2d'].astype(str)
assert not labels['inchikey_2d'].duplicated().any()
labels_by_key = labels.set_index('inchikey_2d', drop=False)

folds_to_run = [0] if DRY_RUN else list(range(N_OUTER_FOLDS))
print('Freeze:', freeze['status'], '| Pretraining:', pretrain['status'])
print('Labels:', len(labels), '| folds:', folds_to_run, '| seeds:', N_SEEDS)

## 2. Token缓存、标签掩码与化合物均衡抽样

训练集的一个`Dataset`元素对应一个化合物，而不是一张谱。每个epoch对该化合物
重新选择一张候选谱，选择概率与冻结的`sample_weight`成正比。验证和测试阶段不随机，
对每张允许的真实谱推理后再按化合物平均。

In [ ]:
class TokenStore:
    def __init__(self, token_dir):
        self.token_dir = Path(token_dir)
        self.caches = {}

    def load(self, cache_name):
        if cache_name not in self.caches:
            path = self.token_dir / cache_name
            assert path.exists(), f'缺少token缓存: {path}'
            self.caches[cache_name] = np.load(path, allow_pickle=False)
        return self.caches[cache_name]

    def get(self, cache_name, row):
        cache = self.load(cache_name)
        row = int(row)
        return {
            'peaks': torch.from_numpy(cache['peak_tokens'][row].astype(np.float32, copy=False)),
            'padding': torch.from_numpy(cache['padding_mask'][row].astype(bool, copy=False)),
            'metadata': torch.from_numpy(cache['metadata_numeric'][row].astype(np.float32, copy=False)),
            'adduct': torch.tensor(int(cache['adduct_code'][row]), dtype=torch.long),
            'instrument': torch.tensor(int(cache['instrument_code'][row]), dtype=torch.long),
            'source': torch.tensor(int(cache['source_code'][row]), dtype=torch.long),
        }

token_store = TokenStore(FREEZE_DIR / 'token_caches')

def label_vector(inchikey_2d, stats, standardized):
    row = labels_by_key.loc[str(inchikey_2d)]
    values, mask = [], []
    for task, spec in TASKS.items():
        value = row[spec['column']]
        available = bool(pd.notna(value))
        mask.append(available)
        if not available:
            values.append(0.0)
        elif spec['kind'] == 'classification' or not standardized:
            values.append(float(value))
        else:
            values.append((float(value) - stats[task]['mean']) / stats[task]['std'])
    return np.asarray(values, np.float32), np.asarray(mask, bool)

def filter_labeled_index(index):
    index = index.copy()
    index['inchikey_2d'] = index['inchikey_2d'].astype(str)
    available_keys = set(labels_by_key.index)
    index = index.loc[index['inchikey_2d'].isin(available_keys)].copy()
    has_any = labels_by_key.loc[index['inchikey_2d'], [x['column'] for x in TASKS.values()]].notna().any(axis=1).to_numpy()
    return index.loc[has_any].reset_index(drop=True)

def deterministic_compound_limit(index, maximum):
    if maximum is None:
        return index.reset_index(drop=True)
    keys = sorted(index['inchikey_2d'].astype(str).unique(), key=lambda x: stable_seed('dry', x))[:maximum]
    return index.loc[index['inchikey_2d'].astype(str).isin(keys)].reset_index(drop=True)

def cap_spectra_per_compound(index, maximum):
    if maximum is None:
        return index.reset_index(drop=True)
    return (
        index.sort_values(['inchikey_2d', 'sample_id'])
        .groupby('inchikey_2d', sort=False, as_index=False)
        .head(maximum).reset_index(drop=True)
    )

class CompoundBalancedDataset(Dataset):
    def __init__(self, index, stats, seed):
        self.index = filter_labeled_index(index)
        self.stats = stats
        self.seed = int(seed)
        self.epoch = 0
        self.compounds = sorted(self.index['inchikey_2d'].unique())
        self.groups = {
            key: group.index.to_numpy(dtype=int)
            for key, group in self.index.groupby('inchikey_2d', sort=False)
        }
        assert len(self.compounds) == self.index['inchikey_2d'].nunique()

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.compounds)

    def __getitem__(self, item):
        key = self.compounds[item]
        positions = self.groups[key]
        candidates = self.index.loc[positions]
        weights = candidates['sample_weight'].to_numpy(float)
        weights = np.clip(weights, 0.0, None)
        probabilities = weights / weights.sum() if weights.sum() > 0 else None
        rng = np.random.default_rng(stable_seed(self.seed, self.epoch, key))
        chosen = int(rng.choice(positions, p=probabilities))
        row = self.index.loc[chosen]
        sample = token_store.get(row['cache_name'], row['cache_row'])
        values, mask = label_vector(key, self.stats, standardized=True)
        sample.update({
            'targets': torch.from_numpy(values),
            'label_mask': torch.from_numpy(mask),
            'inchikey_2d': key,
        })
        return sample

class SpectrumEvaluationDataset(Dataset):
    def __init__(self, index, stats):
        self.index = filter_labeled_index(index)
        self.stats = stats

    def __len__(self):
        return len(self.index)

    def __getitem__(self, item):
        row = self.index.iloc[item]
        sample = token_store.get(row['cache_name'], row['cache_row'])
        values, mask = label_vector(row['inchikey_2d'], self.stats, standardized=False)
        sample.update({
            'targets_original': torch.from_numpy(values),
            'label_mask': torch.from_numpy(mask),
            'inchikey_2d': str(row['inchikey_2d']),
        })
        return sample

def make_loader(dataset, training, seed):
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return DataLoader(
        dataset, batch_size=BATCH_SIZE if training else EVALUATION_BATCH_SIZE,
        shuffle=training, drop_last=False, num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda', generator=generator,
    )

## 3. 与17号严格同构的编码器及多任务预测头

编码器字段和参数名保持一致，B/E检查点采用`strict=True`加载。重建头不参与微调；
归一化的谱图表示进入共享MLP，一次输出5个端点。模型不使用峰位置编码，因此保持集合不变性。

In [ ]:
class SpectrumSetTransformer(nn.Module):
    def __init__(self, config, vocab_sizes):
        super().__init__()
        d_model = config['d_model']
        self.peak_projection = nn.Sequential(
            nn.Linear(config['peak_input_dim'] + 1, d_model),
            nn.GELU(), nn.LayerNorm(d_model),
        )
        self.metadata_projection = nn.Sequential(
            nn.Linear(config['metadata_numeric_dim'], d_model),
            nn.GELU(), nn.LayerNorm(d_model),
        )
        self.adduct_embedding = nn.Embedding(vocab_sizes['adduct'], d_model)
        self.instrument_embedding = nn.Embedding(vocab_sizes['instrument_category'], d_model)
        self.source_embedding = nn.Embedding(vocab_sizes['source_library'], d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=config['n_heads'],
            dim_feedforward=config['dim_feedforward'], dropout=config['dropout'],
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config['n_layers'])
        self.final_norm = nn.LayerNorm(d_model)
        self.reconstruction_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, config['peak_input_dim']),
        )
        self.projection_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, config['projection_dim']),
        )
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, peaks, padding, mask_flags, metadata, adduct, instrument, source):
        peak_input = torch.cat([peaks, mask_flags.unsqueeze(-1)], dim=-1)
        peak_tokens = self.peak_projection(peak_input)
        condition = (
            self.metadata_projection(metadata)
            + self.adduct_embedding(adduct)
            + self.instrument_embedding(instrument)
            + self.source_embedding(source)
        )
        cls = self.cls_token.expand(len(peaks), -1, -1) + condition.unsqueeze(1)
        tokens = torch.cat([cls, peak_tokens], dim=1)
        cls_padding = torch.zeros((len(peaks), 1), dtype=torch.bool, device=padding.device)
        full_padding = torch.cat([cls_padding, padding], dim=1)
        encoded = self.final_norm(self.encoder(tokens, src_key_padding_mask=full_padding))
        spectrum_embedding = F.normalize(self.projection_head(encoded[:, 0]), dim=-1)
        reconstructed_peaks = self.reconstruction_head(encoded[:, 1:])
        return spectrum_embedding, reconstructed_peaks

class MultiTaskSpectrumModel(nn.Module):
    def __init__(self, config, vocab_sizes, n_tasks):
        super().__init__()
        self.encoder = SpectrumSetTransformer(config, vocab_sizes)
        for parameter in self.encoder.reconstruction_head.parameters():
            parameter.requires_grad = False
        self.head = nn.Sequential(
            nn.LayerNorm(config['projection_dim']),
            nn.Linear(config['projection_dim'], config['projection_dim']),
            nn.GELU(), nn.Dropout(HEAD_DROPOUT),
            nn.Linear(config['projection_dim'], n_tasks),
        )

    def forward(self, peaks, padding, metadata, adduct, instrument, source):
        mask_flags = torch.zeros(peaks.shape[:2], dtype=peaks.dtype, device=peaks.device)
        embedding, _ = self.encoder(peaks, padding, mask_flags, metadata, adduct, instrument, source)
        return self.head(embedding)

VOCAB_SIZES = {
    'adduct': len(vocab['adduct']),
    'instrument_category': len(vocab['instrument_category']),
    'source_library': len(vocab['source_library']),
}

def trainable_parameter_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

example_model = MultiTaskSpectrumModel(MODEL_CONFIG, VOCAB_SIZES, len(TASKS))
print('Trainable parameters:', trainable_parameter_count(example_model))
del example_model

## 4. 掩码多任务损失、化合物级验证与正式指标

每个batch内，各端点先仅在有标签的化合物上计算损失，再对当前可用端点等权平均。
这防止标签较多的P或某个T端点压制样本较少的B端点。回归训练使用标准化SmoothL1；
最佳epoch按真实内层验证谱聚合后的等权复合损失选择。

In [ ]:
def move_batch(batch, target_device):
    return {
        key: value.to(target_device, non_blocking=True) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }

def masked_multitask_loss(outputs, targets, label_mask):
    losses = []
    details = {}
    for task_index, (task, spec) in enumerate(TASKS.items()):
        available = label_mask[:, task_index]
        if not bool(available.any()):
            details[task] = np.nan
            continue
        prediction = outputs[available, task_index]
        truth = targets[available, task_index]
        if spec['kind'] == 'classification':
            task_loss = F.binary_cross_entropy_with_logits(prediction, truth)
        else:
            task_loss = F.smooth_l1_loss(prediction, truth, beta=0.5)
        losses.append(task_loss)
        details[task] = float(task_loss.detach())
    if not losses:
        raise RuntimeError('当前batch没有任何可用标签。')
    return torch.stack(losses).mean(), details

def run_training_epoch(model, loader, optimizer, scaler):
    model.train()
    total_weighted = 0.0
    total_n = 0
    task_sums = {task: 0.0 for task in TASKS}
    task_counts = {task: 0 for task in TASKS}
    for batch in tqdm(loader, desc='train', leave=False):
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        context = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled)
        with context:
            outputs = model(
                batch['peaks'], batch['padding'], batch['metadata'],
                batch['adduct'], batch['instrument'], batch['source'],
            )
            loss, details = masked_multitask_loss(outputs, batch['targets'], batch['label_mask'])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        batch_n = len(batch['targets'])
        total_weighted += float(loss.detach()) * batch_n
        total_n += batch_n
        for task, value in details.items():
            if np.isfinite(value):
                task_sums[task] += value
                task_counts[task] += 1
    result = {'total_loss': total_weighted / max(total_n, 1)}
    result.update({
        f'{task}_loss': task_sums[task] / task_counts[task] if task_counts[task] else np.nan
        for task in TASKS
    })
    return result

@torch.no_grad()
def predict_compounds(model, index, stats, split_role, outer_fold, route, seed_index):
    dataset = SpectrumEvaluationDataset(index, stats)
    loader = make_loader(dataset, training=False, seed=stable_seed('eval', outer_fold, route, seed_index, split_role))
    accumulator = {}
    model.eval()
    for batch in tqdm(loader, desc=split_role, leave=False):
        keys = list(batch['inchikey_2d'])
        targets = batch['targets_original'].numpy()
        masks = batch['label_mask'].numpy().astype(bool)
        batch_device = move_batch({k: v for k, v in batch.items() if k != 'inchikey_2d'}, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            raw = model(
                batch_device['peaks'], batch_device['padding'], batch_device['metadata'],
                batch_device['adduct'], batch_device['instrument'], batch_device['source'],
            ).float().cpu().numpy()
        for row_index, key in enumerate(keys):
            for task_index, (task, spec) in enumerate(TASKS.items()):
                if not masks[row_index, task_index]:
                    continue
                if spec['kind'] == 'classification':
                    value = float(1.0 / (1.0 + np.exp(-np.clip(raw[row_index, task_index], -40, 40))))
                else:
                    value = float(raw[row_index, task_index] * stats[task]['std'] + stats[task]['mean'])
                record = accumulator.setdefault((key, task), {
                    'prediction_sum': 0.0, 'n_spectra': 0,
                    'y_true': float(targets[row_index, task_index]),
                })
                record['prediction_sum'] += value
                record['n_spectra'] += 1

    rows = []
    for (key, task), record in accumulator.items():
        value = record['prediction_sum'] / record['n_spectra']
        kind = TASKS[task]['kind']
        rows.append({
            'outer_fold': int(outer_fold), 'route': route, 'seed_index': int(seed_index),
            'split_role': split_role, 'inchikey_2d': key, 'task': task, 'kind': kind,
            'y_true': record['y_true'],
            'y_prediction': int(value >= 0.5) if kind == 'classification' else value,
            'y_probability': value if kind == 'classification' else np.nan,
            'n_spectra': int(record['n_spectra']),
        })
    return pd.DataFrame(rows)

def validation_composite_loss(predictions, stats):
    task_losses = []
    for task, spec in TASKS.items():
        part = predictions.loc[predictions['task'].eq(task)]
        if part.empty:
            continue
        y = part['y_true'].to_numpy(float)
        if spec['kind'] == 'classification':
            p = np.clip(part['y_probability'].to_numpy(float), 1e-6, 1 - 1e-6)
            value = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        else:
            prediction = part['y_prediction'].to_numpy(float)
            value = np.mean(((prediction - y) / stats[task]['std']) ** 2)
        task_losses.append(float(value))
    if len(task_losses) != len(TASKS):
        raise RuntimeError(f'内层验证缺少端点: {len(task_losses)}/{len(TASKS)}')
    return float(np.mean(task_losses))

def metric_record(frame):
    kind = frame['kind'].iloc[0]
    y_true = frame['y_true'].to_numpy(float)
    if kind == 'classification':
        probability = frame['y_probability'].to_numpy(float)
        both = np.unique(y_true).size == 2
        return {
            'n': len(frame), 'n_positive': int(y_true.sum()),
            'prevalence': float(y_true.mean()),
            'pr_auc': float(average_precision_score(y_true, probability)) if both else np.nan,
            'roc_auc': float(roc_auc_score(y_true, probability)) if both else np.nan,
            'brier': float(brier_score_loss(y_true, probability)),
            'balanced_accuracy_0.5': float(balanced_accuracy_score(y_true, probability >= 0.5)) if both else np.nan,
            'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'spearman': np.nan,
        }
    prediction = frame['y_prediction'].to_numpy(float)
    rho = spearmanr(y_true, prediction).statistic if len(frame) >= 3 and np.unique(prediction).size > 1 else np.nan
    return {
        'n': len(frame), 'n_positive': np.nan, 'prevalence': np.nan,
        'pr_auc': np.nan, 'roc_auc': np.nan, 'brier': np.nan,
        'balanced_accuracy_0.5': np.nan,
        'mae': float(mean_absolute_error(y_true, prediction)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, prediction))),
        'r2': float(r2_score(y_true, prediction)), 'spearman': float(rho),
    }

## 5. 运行90个折内多任务微调

每个run均保存`last`和`best`检查点、外层预测文件和独立run清单。中断后保持配置不变重新运行，
已完成run直接跳过，未完成run从最近epoch继续。B/E严格加载与当前外折一致的17号检查点；
A/C/D/F保持随机初始化。外层测试在最佳epoch冻结后只运行一次。

In [ ]:
HISTORY_PATH = OUTPUT_DIR / 'transformer_multitask_history.csv'
RUN_RECORD_PATH = OUTPUT_DIR / 'transformer_multitask_run_records.csv'
if HISTORY_PATH.exists():
    history_rows = pd.read_csv(HISTORY_PATH).to_dict('records')
else:
    history_rows = []

def route_finetuning_path(outer_fold, route):
    return FREEZE_DIR / 'fold_indices' / f'outer_{outer_fold}' / f'{route}__finetuning.csv.gz'

def evaluation_path(outer_fold):
    return FREEZE_DIR / 'fold_indices' / f'outer_{outer_fold}' / 'real_evaluation_index.csv.gz'

def pretraining_checkpoint(outer_fold, route):
    if route == 'B_real_pretrain':
        name = 'B_real_pretrain'
    elif route == 'E_cfm_pretrain_real_finetune':
        name = 'E_cfm_pretrain_real_finetune'
    else:
        return None
    return PRETRAIN_DIR / 'checkpoints' / f'outer_{outer_fold}' / f'{name}__best.pt'

def regression_stats_from_common_real_train(outer_fold):
    common = pd.read_csv(route_finetuning_path(outer_fold, 'A_real_scratch'))
    keys = common['inchikey_2d'].astype(str).unique()
    rows = labels_by_key.reindex(keys)
    stats = {}
    for task, spec in TASKS.items():
        if spec['kind'] == 'classification':
            stats[task] = {'mean': 0.0, 'std': 1.0}
        else:
            values = rows[spec['column']].dropna().to_numpy(float)
            std = float(np.std(values, ddof=0))
            stats[task] = {'mean': float(np.mean(values)), 'std': max(std, 1e-6)}
    return stats

def load_pretraining_if_required(model, outer_fold, route):
    checkpoint_path = pretraining_checkpoint(outer_fold, route)
    if checkpoint_path is None:
        return 'scratch', None
    assert checkpoint_path.exists(), f'缺少17号检查点: {checkpoint_path}'
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    assert int(saved['metadata']['outer_fold']) == int(outer_fold)
    assert saved['metadata']['route'] == route
    model.encoder.load_state_dict(saved['model_state'], strict=True)
    initialization = 'pretrained_real' if route == 'B_real_pretrain' else 'pretrained_real_plus_cfm'
    return initialization, sha256_file(checkpoint_path)

def collect_run_records():
    records = []
    for path in sorted(RUN_MANIFEST_DIR.glob('outer_*/*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            records.append(payload['record'])
    return pd.DataFrame(records)

for outer_fold in folds_to_run:
    stats = regression_stats_from_common_real_train(outer_fold)
    eval_full = pd.read_csv(evaluation_path(outer_fold))
    eval_full['inchikey_2d'] = eval_full['inchikey_2d'].astype(str)
    validation_full = eval_full.loc[eval_full['split_role'].eq('inner_validation')].drop(columns='split_role')
    test_full = eval_full.loc[eval_full['split_role'].eq('outer_test')].drop(columns='split_role')
    validation_full = filter_labeled_index(validation_full)
    test_full = filter_labeled_index(test_full)
    validation_index = cap_spectra_per_compound(
        validation_full, MAX_VALIDATION_SPECTRA_PER_COMPOUND,
    )
    if DRY_RUN:
        validation_index = deterministic_compound_limit(validation_index, 16)
        test_index = deterministic_compound_limit(cap_spectra_per_compound(test_full, 2), 16)
    else:
        test_index = test_full

    for route in ROUTES:
        train_path = route_finetuning_path(outer_fold, route)
        assert train_path.exists(), train_path
        train_full = filter_labeled_index(pd.read_csv(train_path))
        train_index = deterministic_compound_limit(train_full, 48) if DRY_RUN else train_full

        train_keys = set(train_index['inchikey_2d'])
        validation_keys = set(validation_index['inchikey_2d'])
        test_keys = set(test_index['inchikey_2d'])
        assert train_keys.isdisjoint(validation_keys)
        assert train_keys.isdisjoint(test_keys)
        assert validation_keys.isdisjoint(test_keys)

        for seed_index in range(N_SEEDS):
            run_seed = BASE_SEED + outer_fold * 100 + seed_index
            run_name = f'{route}__seed_{seed_index}'
            checkpoint_subdir = CHECKPOINT_DIR / f'outer_{outer_fold}'
            prediction_subdir = PREDICTION_DIR / f'outer_{outer_fold}'
            manifest_subdir = RUN_MANIFEST_DIR / f'outer_{outer_fold}'
            for directory in [checkpoint_subdir, prediction_subdir, manifest_subdir]:
                directory.mkdir(parents=True, exist_ok=True)
            best_path = checkpoint_subdir / f'{run_name}__best.pt'
            last_path = checkpoint_subdir / f'{run_name}__last.pt'
            prediction_path = prediction_subdir / f'{run_name}.csv.gz'
            run_manifest_path = manifest_subdir / f'{run_name}.json'

            pretrain_path = pretraining_checkpoint(outer_fold, route)
            pretrain_hash = sha256_file(pretrain_path) if pretrain_path is not None else None
            expected_metadata = {
                'outer_fold': outer_fold, 'route': route, 'seed_index': seed_index,
                'run_seed': run_seed,
                'freeze_manifest_sha256': INPUT_HASHES['freeze_manifest'],
                'pretraining_manifest_sha256': INPUT_HASHES['pretraining_manifest'],
                'labels_sha256': INPUT_HASHES['labels'],
                'finetuning_index_sha256': sha256_file(train_path),
                'evaluation_index_sha256': sha256_file(evaluation_path(outer_fold)),
                'pretraining_checkpoint_sha256': pretrain_hash,
                'task_schema': TASKS,
                'regression_stats': stats,
                'model_config': MODEL_CONFIG,
                'vocab_sizes': VOCAB_SIZES,
                'training_config': {
                    'max_epochs': MAX_EPOCHS, 'min_epochs': MIN_EPOCHS,
                    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
                    'batch_size': BATCH_SIZE,
                    'encoder_learning_rate': ENCODER_LEARNING_RATE,
                    'head_learning_rate': HEAD_LEARNING_RATE,
                    'weight_decay': WEIGHT_DECAY,
                    'compound_equal_sampling': True,
                    'validation_spectra_per_compound': MAX_VALIDATION_SPECTRA_PER_COMPOUND,
                },
            }

            if run_manifest_path.exists() and not RETRAIN_COMPLETED:
                completed = json.loads(run_manifest_path.read_text(encoding='utf-8'))
                if completed.get('metadata') != expected_metadata:
                    raise RuntimeError(f'已完成run与当前冻结配置不一致: {run_manifest_path}')
                assert completed['status'] == 'completed'
                assert best_path.exists() and prediction_path.exists()
                assert sha256_file(best_path) == completed['record']['best_checkpoint_sha256']
                assert sha256_file(prediction_path) == completed['record']['prediction_sha256']
                print('Skip completed:', outer_fold, route, seed_index)
                continue

            set_seed(run_seed)
            model = MultiTaskSpectrumModel(MODEL_CONFIG, VOCAB_SIZES, len(TASKS)).to(device)
            initialization, loaded_pretrain_hash = load_pretraining_if_required(model, outer_fold, route)
            assert loaded_pretrain_hash == pretrain_hash
            train_dataset = CompoundBalancedDataset(train_index, stats, run_seed)
            optimizer = torch.optim.AdamW([
                {'params': [p for p in model.encoder.parameters() if p.requires_grad], 'lr': ENCODER_LEARNING_RATE},
                {'params': model.head.parameters(), 'lr': HEAD_LEARNING_RATE},
            ], weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
            scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
            start_epoch = 0
            best_epoch = -1
            best_validation_loss = np.inf
            stale_epochs = 0
            elapsed_before = 0.0

            if last_path.exists() and RESUME_INCOMPLETE and not RETRAIN_COMPLETED:
                saved = torch.load(last_path, map_location=device, weights_only=False)
                if saved.get('metadata') != expected_metadata:
                    raise RuntimeError(f'未完成检查点与当前配置不一致: {last_path}')
                model.load_state_dict(saved['model_state'])
                optimizer.load_state_dict(saved['optimizer_state'])
                scheduler.load_state_dict(saved['scheduler_state'])
                scaler.load_state_dict(saved['scaler_state'])
                start_epoch = int(saved['epoch']) + 1
                best_epoch = int(saved['best_epoch'])
                best_validation_loss = float(saved['best_validation_loss'])
                stale_epochs = int(saved['stale_epochs'])
                elapsed_before = float(saved.get('elapsed_seconds', 0.0))
                print('Resume:', outer_fold, route, seed_index, 'epoch', start_epoch)

            if not RUN_TRAINING:
                print('Training disabled:', outer_fold, route, seed_index)
                continue

            started = time.perf_counter()
            completed_epoch = start_epoch - 1
            for epoch in range(start_epoch, MAX_EPOCHS):
                train_dataset.set_epoch(epoch)
                train_loader = make_loader(train_dataset, training=True, seed=run_seed + epoch * 1009)
                train_metrics = run_training_epoch(model, train_loader, optimizer, scaler)
                validation_predictions = predict_compounds(
                    model, validation_index, stats, 'inner_validation', outer_fold, route, seed_index,
                )
                validation_loss = validation_composite_loss(validation_predictions, stats)
                validation_task_metrics = {
                    task: metric_record(validation_predictions.loc[validation_predictions['task'].eq(task)])
                    for task in TASKS
                }
                scheduler.step()
                completed_epoch = epoch
                row = {
                    'outer_fold': outer_fold, 'route': route, 'seed_index': seed_index,
                    'epoch': epoch, 'train_compound_n': len(train_dataset),
                    'encoder_learning_rate': optimizer.param_groups[0]['lr'],
                    'head_learning_rate': optimizer.param_groups[1]['lr'],
                    'validation_composite_loss': validation_loss,
                    **{f'train_{key}': value for key, value in train_metrics.items()},
                }
                history_rows = [item for item in history_rows if not (
                    int(item.get('outer_fold', -1)) == outer_fold
                    and str(item.get('route')) == route
                    and int(item.get('seed_index', -1)) == seed_index
                    and int(item.get('epoch', -1)) == epoch
                )]
                history_rows.append(row)
                history_frame = pd.DataFrame(history_rows).sort_values(['outer_fold', 'route', 'seed_index', 'epoch'])
                atomic_csv(history_frame, HISTORY_PATH)

                improved = validation_loss < best_validation_loss - 1e-8
                if improved:
                    best_validation_loss = validation_loss
                    best_epoch = epoch
                    stale_epochs = 0
                    save_torch_atomic({
                        'metadata': expected_metadata, 'epoch': epoch,
                        'model_state': {k: v.detach().cpu() for k, v in model.state_dict().items()},
                        'validation_composite_loss': validation_loss,
                        'validation_task_metrics': validation_task_metrics,
                    }, best_path)
                else:
                    stale_epochs += 1

                elapsed = elapsed_before + time.perf_counter() - started
                save_torch_atomic({
                    'metadata': expected_metadata, 'epoch': epoch,
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'scheduler_state': scheduler.state_dict(),
                    'scaler_state': scaler.state_dict(),
                    'best_epoch': best_epoch,
                    'best_validation_loss': best_validation_loss,
                    'stale_epochs': stale_epochs, 'elapsed_seconds': elapsed,
                }, last_path)
                print(
                    f'fold={outer_fold} route={route} seed={seed_index} epoch={epoch} '
                    f'train={train_metrics["total_loss"]:.5f} val={validation_loss:.5f} '
                    f'best={best_validation_loss:.5f}'
                )
                if epoch + 1 >= MIN_EPOCHS and stale_epochs >= EARLY_STOPPING_PATIENCE:
                    break

            assert best_path.exists()
            best_saved = torch.load(best_path, map_location='cpu', weights_only=False)
            model.load_state_dict(best_saved['model_state'])
            model.to(device)
            outer_predictions = predict_compounds(
                model, test_index, stats, 'outer_test', outer_fold, route, seed_index,
            )
            atomic_csv(outer_predictions, prediction_path, compression='gzip')
            total_elapsed = elapsed_before + time.perf_counter() - started
            source_counts = train_index['source_pool'].value_counts().to_dict()
            record = {
                'outer_fold': outer_fold, 'route': route, 'seed_index': seed_index,
                'status': 'completed', 'run_seed': run_seed,
                'initialization': initialization,
                'train_spectra_n': len(train_index),
                'train_compound_n': train_index['inchikey_2d'].nunique(),
                'train_real_spectra_n': int(source_counts.get('development_labeled_real', 0)),
                'train_cfm_spectra_n': int(source_counts.get('cfm', 0)),
                'validation_spectra_n': len(validation_index),
                'validation_compound_n': validation_index['inchikey_2d'].nunique(),
                'outer_test_spectra_n': len(test_index),
                'outer_test_compound_n': test_index['inchikey_2d'].nunique(),
                'train_validation_identity_overlap': len(train_keys & validation_keys),
                'train_test_identity_overlap': len(train_keys & test_keys),
                'validation_test_identity_overlap': len(validation_keys & test_keys),
                'best_epoch': int(best_saved['epoch']),
                'epochs_completed': completed_epoch + 1,
                'best_validation_composite_loss': float(best_saved['validation_composite_loss']),
                'runtime_minutes': total_elapsed / 60.0,
                'best_checkpoint': str(best_path.relative_to(OUTPUT_DIR)),
                'best_checkpoint_sha256': sha256_file(best_path),
                'prediction_file': str(prediction_path.relative_to(OUTPUT_DIR)),
                'prediction_sha256': sha256_file(prediction_path),
                'device': str(device),
            }
            atomic_json({
                'status': 'completed', 'completed_at_utc': utc_now(),
                'metadata': expected_metadata, 'record': record,
            }, run_manifest_path)
            records_frame = collect_run_records()
            atomic_csv(records_frame.sort_values(['outer_fold', 'route', 'seed_index']), RUN_RECORD_PATH)

run_records = collect_run_records()
if len(run_records):
    run_records = run_records.sort_values(['outer_fold', 'route', 'seed_index']).reset_index(drop=True)
    atomic_csv(run_records, RUN_RECORD_PATH)
display(run_records)

## 6. 外层OOF预测、三种子集成与A–F性能

先保留每个种子的完整外层预测，再对同一“路线×外折×端点×化合物”的三个种子取均值。
主结果使用三种子集成；同时输出逐种子和逐折指标，供19号进行配对bootstrap与稳定性分析。

In [ ]:
expected_runs = len(folds_to_run) * len(ROUTES) * N_SEEDS
prediction_frames = []
for path in sorted(PREDICTION_DIR.glob('outer_*/*.csv.gz')):
    frame = pd.read_csv(path)
    if int(frame['outer_fold'].iloc[0]) in folds_to_run:
        prediction_frames.append(frame)
seed_predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()

seed_metric_rows = []
fold_seed_metric_rows = []
if len(seed_predictions):
    for (route, seed_index, task), frame in seed_predictions.groupby(['route', 'seed_index', 'task'], sort=False):
        seed_metric_rows.append({
            'route': route, 'seed_index': int(seed_index), 'task': task,
            'kind': frame['kind'].iloc[0], **metric_record(frame),
        })
    for (outer_fold, route, seed_index, task), frame in seed_predictions.groupby(
        ['outer_fold', 'route', 'seed_index', 'task'], sort=False,
    ):
        fold_seed_metric_rows.append({
            'outer_fold': int(outer_fold), 'route': route,
            'seed_index': int(seed_index), 'task': task,
            'kind': frame['kind'].iloc[0], **metric_record(frame),
        })
seed_metrics = pd.DataFrame(seed_metric_rows)
fold_seed_metrics = pd.DataFrame(fold_seed_metric_rows)

ensemble_keys = ['outer_fold', 'route', 'split_role', 'inchikey_2d', 'task', 'kind']
ensemble = (
    seed_predictions.groupby(ensemble_keys, as_index=False)
    .agg(
        y_true=('y_true', 'first'),
        y_prediction=('y_prediction', 'mean'),
        y_probability=('y_probability', 'mean'),
        n_spectra=('n_spectra', 'first'),
        seed_n=('seed_index', 'nunique'),
    )
) if len(seed_predictions) else pd.DataFrame()
if len(ensemble):
    classification_rows = ensemble['kind'].eq('classification')
    ensemble.loc[classification_rows, 'y_prediction'] = (
        ensemble.loc[classification_rows, 'y_probability'] >= 0.5
    ).astype(int)

pooled_metric_rows, fold_metric_rows = [], []
if len(ensemble):
    for (route, task), frame in ensemble.groupby(['route', 'task'], sort=False):
        pooled_metric_rows.append({
            'route': route, 'task': task, 'kind': frame['kind'].iloc[0],
            'model': 'three_seed_ensemble', **metric_record(frame),
        })
    for (outer_fold, route, task), frame in ensemble.groupby(['outer_fold', 'route', 'task'], sort=False):
        fold_metric_rows.append({
            'outer_fold': int(outer_fold), 'route': route, 'task': task,
            'kind': frame['kind'].iloc[0], 'model': 'three_seed_ensemble',
            **metric_record(frame),
        })
pooled_metrics = pd.DataFrame(pooled_metric_rows)
fold_metrics = pd.DataFrame(fold_metric_rows)

atomic_csv(seed_predictions, OUTPUT_DIR / 'transformer_outer_test_predictions_by_seed.csv.gz', compression='gzip')
atomic_csv(ensemble, OUTPUT_DIR / 'transformer_outer_test_predictions_ensemble.csv.gz', compression='gzip')
atomic_csv(seed_metrics, OUTPUT_DIR / 'transformer_metrics_by_seed.csv')
atomic_csv(fold_seed_metrics, OUTPUT_DIR / 'transformer_metrics_by_fold_and_seed.csv')
atomic_csv(pooled_metrics, OUTPUT_DIR / 'transformer_metrics_ensemble.csv')
atomic_csv(fold_metrics, OUTPUT_DIR / 'transformer_metrics_by_fold_ensemble.csv')
display(pooled_metrics)

## 7. 泄漏、路线、检查点与预测完整性QC

这里审核的是科学边界而非仅检查文件存在：训练/验证/测试身份交集必须为0；
B/E必须加载对应折预训练，其他路线必须scratch；所有路线在同一折同一端点的真实测试
化合物集合必须完全一致；每个主预测必须恰有3个种子。

In [ ]:
expected_initialization = {
    'A_real_scratch': 'scratch',
    'B_real_pretrain': 'pretrained_real',
    'C_same_compound_cfm': 'scratch',
    'D_synthetic_new_cfm': 'scratch',
    'E_cfm_pretrain_real_finetune': 'pretrained_real_plus_cfm',
    'F_cfm_only_negative_control': 'scratch',
}

checkpoint_hash_ok = True
prediction_hash_ok = True
for _, row in run_records.iterrows():
    checkpoint = OUTPUT_DIR / row['best_checkpoint']
    prediction = OUTPUT_DIR / row['prediction_file']
    checkpoint_hash_ok &= checkpoint.exists() and sha256_file(checkpoint) == row['best_checkpoint_sha256']
    prediction_hash_ok &= prediction.exists() and sha256_file(prediction) == row['prediction_sha256']

identity_disjoint = bool(
    len(run_records) == expected_runs
    and run_records[[
        'train_validation_identity_overlap',
        'train_test_identity_overlap',
        'validation_test_identity_overlap',
    ]].astype(int).eq(0).all().all()
)
initialization_correct = bool(
    len(run_records) == expected_runs
    and run_records.apply(
        lambda row: row['initialization'] == expected_initialization[row['route']], axis=1,
    ).all()
)
seed_complete = bool(
    len(ensemble)
    and ensemble['seed_n'].astype(int).eq(N_SEEDS).all()
)
prediction_finite = bool(
    len(ensemble)
    and np.isfinite(ensemble['y_true'].to_numpy(float)).all()
    and np.isfinite(ensemble['y_prediction'].to_numpy(float)).all()
    and np.isfinite(
        ensemble.loc[ensemble['kind'].eq('classification'), 'y_probability'].to_numpy(float)
    ).all()
)

test_identity_equal = True
identity_details = []
if len(ensemble):
    for (outer_fold, task), frame in ensemble.groupby(['outer_fold', 'task']):
        route_sets = {
            route: set(part['inchikey_2d'])
            for route, part in frame.groupby('route')
        }
        equal = len(route_sets) == len(ROUTES) and len({tuple(sorted(x)) for x in route_sets.values()}) == 1
        test_identity_equal &= equal
        identity_details.append({
            'outer_fold': int(outer_fold), 'task': task,
            'routes_present': len(route_sets), 'identity_sets_equal': bool(equal),
            'test_compound_n': len(next(iter(route_sets.values()))) if route_sets else 0,
        })
identity_audit = pd.DataFrame(identity_details)
atomic_csv(identity_audit, OUTPUT_DIR / 'transformer_outer_test_identity_audit.csv')

expected_pooled_rows = len(ROUTES) * len(TASKS)
metric_finite = bool(
    len(pooled_metrics) == expected_pooled_rows
    and np.isfinite(
        pooled_metrics.apply(
            lambda row: row['pr_auc'] if row['kind'] == 'classification' else row['r2'], axis=1,
        ).to_numpy(float)
    ).all()
)
source_rules = bool(
    len(run_records) == expected_runs
    and run_records.loc[run_records['route'].isin(['A_real_scratch', 'B_real_pretrain', 'E_cfm_pretrain_real_finetune']), 'train_cfm_spectra_n'].astype(int).eq(0).all()
    and run_records.loc[run_records['route'].eq('F_cfm_only_negative_control'), 'train_real_spectra_n'].astype(int).eq(0).all()
    and run_records.loc[run_records['route'].isin(['C_same_compound_cfm', 'D_synthetic_new_cfm']), 'train_cfm_spectra_n'].astype(int).gt(0).all()
)

qc = pd.DataFrame([
    {'check': 'freeze_and_pretraining_ready', 'passed': freeze['status'] == 'READY_FOR_NOTEBOOK_17' and pretrain['status'] == 'READY_FOR_NOTEBOOK_18', 'detail': f"{freeze['status']} | {pretrain['status']}"},
    {'check': 'chemvista_remains_locked', 'passed': freeze['external_test_unlocked'] is False and freeze['chemvista_spectra_used'] is False, 'detail': 'no ChemVista path loaded'},
    {'check': 'all_90_runs_completed', 'passed': len(run_records) == expected_runs, 'detail': f'{len(run_records)}/{expected_runs}'},
    {'check': 'train_validation_test_identities_disjoint', 'passed': identity_disjoint, 'detail': 'all run intersections must be 0'},
    {'check': 'pretraining_initialization_by_route_correct', 'passed': initialization_correct, 'detail': str(expected_initialization)},
    {'check': 'route_source_composition_correct', 'passed': source_rules, 'detail': 'A/B/E real; C/D mixed; F CFM only'},
    {'check': 'checkpoint_hashes_match', 'passed': checkpoint_hash_ok and len(run_records) == expected_runs, 'detail': len(run_records)},
    {'check': 'prediction_hashes_match', 'passed': prediction_hash_ok and len(run_records) == expected_runs, 'detail': len(run_records)},
    {'check': 'three_seed_ensemble_complete', 'passed': seed_complete, 'detail': f'seed_n={N_SEEDS}'},
    {'check': 'outer_test_identity_sets_equal_across_routes', 'passed': test_identity_equal and len(identity_audit) == len(folds_to_run) * len(TASKS), 'detail': len(identity_audit)},
    {'check': 'outer_test_predictions_finite', 'passed': prediction_finite, 'detail': len(ensemble)},
    {'check': 'all_route_endpoint_primary_metrics_finite', 'passed': metric_finite, 'detail': f'{len(pooled_metrics)}/{expected_pooled_rows}'},
    {'check': 'compound_equal_training_sampler_used', 'passed': True, 'detail': 'one dataset item per unique IK2D per epoch'},
])
atomic_csv(qc, OUTPUT_DIR / 'transformer_multitask_qc.csv')
display(identity_audit)
display(qc)

## 8. 非交互式诊断图与最终清单

P面板显示PR-AUC，四个回归面板显示R²。这里只展示内部Development外折结果，
不据此接触ChemVista。正式的配对置信区间、路线选择和与结构/SIRIUS/DreaMS基线的
统一比较放在19号完成。

In [ ]:
if len(pooled_metrics):
    route_short = {route: route.split('_', 1)[0] for route in ROUTES}
    fig, axes = plt.subplots(2, 3, figsize=(15.5, 9.0))
    axes = axes.ravel()
    colors = ['#4C78A8', '#F58518', '#54A24B', '#E45756', '#72B7B2', '#B279A2']
    for axis, task in zip(axes, TASK_NAMES):
        part = pooled_metrics.loc[pooled_metrics['task'].eq(task)].set_index('route').reindex(ROUTES)
        metric = 'pr_auc' if TASKS[task]['kind'] == 'classification' else 'r2'
        values = part[metric].to_numpy(float)
        axis.bar([route_short[x] for x in ROUTES], values, color=colors)
        axis.axhline(0.0, color='black', linewidth=0.8)
        axis.set_title(f'{task}\n{metric.upper()}')
        axis.set_xlabel('A–F route')
        for position, value in enumerate(values):
            axis.text(position, value, f'{value:.3f}', ha='center', va='bottom' if value >= 0 else 'top', fontsize=8)
    axes[-1].axis('off')
    fig.suptitle('Transformer A–F ablation: real outer-fold test, three-seed ensemble', y=1.01)
    fig.tight_layout()
    figure_path = FIGURE_DIR / '18_transformer_ablation_primary_metrics.png'
    fig.savefig(figure_path, dpi=240, bbox_inches='tight')
    plt.close(fig)
    print('Figure saved:', figure_path)

all_qc_pass = bool(as_bool(qc['passed']).all())
if DRY_RUN:
    FINETUNING_STATUS = 'DRY_RUN_VALIDATION_ONLY'
elif not RUN_TRAINING and len(run_records) < expected_runs:
    FINETUNING_STATUS = 'FINETUNING_INCOMPLETE_RESUME'
elif all_qc_pass:
    FINETUNING_STATUS = 'READY_FOR_NOTEBOOK_19'
else:
    FINETUNING_STATUS = 'BLOCKED_REVIEW_QC'

summary_outputs = [
    HISTORY_PATH, RUN_RECORD_PATH,
    OUTPUT_DIR / 'transformer_outer_test_predictions_by_seed.csv.gz',
    OUTPUT_DIR / 'transformer_outer_test_predictions_ensemble.csv.gz',
    OUTPUT_DIR / 'transformer_metrics_by_seed.csv',
    OUTPUT_DIR / 'transformer_metrics_by_fold_and_seed.csv',
    OUTPUT_DIR / 'transformer_metrics_ensemble.csv',
    OUTPUT_DIR / 'transformer_metrics_by_fold_ensemble.csv',
    OUTPUT_DIR / 'transformer_outer_test_identity_audit.csv',
    OUTPUT_DIR / 'transformer_multitask_qc.csv',
    FIGURE_DIR / '18_transformer_ablation_primary_metrics.png',
]
summary_outputs = [path for path in summary_outputs if path.exists()]
manifest = {
    'notebook': '18_Transformer多任务微调与A-F消融.ipynb',
    'generated_at_utc': utc_now(),
    'status': FINETUNING_STATUS,
    'scientific_scope': 'Development-only nested outer-fold A-F transformer ablation',
    'external_test_unlocked': False,
    'chemvista_used': False,
    'selection_boundary': 'inner validation only; outer test evaluated after best epoch',
    'task_schema': TASKS,
    'routes': route_spec['routes'],
    'training': {
        'outer_folds': folds_to_run, 'n_seeds': N_SEEDS,
        'expected_runs': expected_runs, 'completed_runs': len(run_records),
        'multi_task_missing_label_mask': True,
        'compound_equal_sampling': True,
        'max_epochs': MAX_EPOCHS, 'minimum_epochs': MIN_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'encoder_learning_rate': ENCODER_LEARNING_RATE,
        'head_learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
    },
    'device': {
        'type': str(device),
        'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
        'mixed_precision': amp_enabled,
    },
    'model': {
        **MODEL_CONFIG, 'head_dropout': HEAD_DROPOUT,
        'trainable_parameter_count': trainable_parameter_count(
            MultiTaskSpectrumModel(MODEL_CONFIG, VOCAB_SIZES, len(TASKS))
        ),
    },
    'input_hashes': INPUT_HASHES,
    'summary_output_hashes': {
        str(path.relative_to(OUTPUT_DIR)): sha256_file(path) for path in summary_outputs
    },
    'checkpoint_and_prediction_hashes': 'stored and verified per run in run_manifests/',
    'qc_all_pass': all_qc_pass,
    'software': {
        'python': sys.version, 'torch': torch.__version__,
        'numpy': np.__version__, 'pandas': pd.__version__,
        'platform': platform.platform(),
    },
}
atomic_json(manifest, OUTPUT_DIR / 'transformer_finetuning_manifest.json')
print('FINETUNING_STATUS:', FINETUNING_STATUS)
print('Manifest:', OUTPUT_DIR / 'transformer_finetuning_manifest.json')
if FINETUNING_STATUS == 'READY_FOR_NOTEBOOK_19':
    print('18号完成：进入19号内部统计比较、路线冻结与ChemVista解锁门控。')
elif FINETUNING_STATUS == 'DRY_RUN_VALIDATION_ONLY':
    print('这只是DRY_RUN；改回DRY_RUN=False后运行完整90个run。')
else:
    print('结果尚未满足全部QC；保持配置不变重新运行可续跑。')

## 下一步

19号读取18号三种子OOF预测，完成：

1. A–F对A路线的化合物级配对bootstrap差值与95%CI；
2. 与Morgan/MACCS、简单MS/MS、Set Transformer、DreaMS、SIRIUS基线按共同化合物统一比较；
3. 依据预注册主指标和稳定性冻结最终路线、随机种子集成方式及风险阈值；
4. 只有19号全部内部QC通过后，才生成一次性ChemVista盲测输入。